# Raw Data Exploration

In [0]:
file_location = "s3://my-datasets-for-demo/customer-churn-datasets/converted/online_retail_II_2009-2010.csv"
df = spark.read.csv(file_location, header=True, inferSchema=True)
df.show(10)

In [0]:
file_location_2 = "s3://my-datasets-for-demo/customer-churn-datasets/converted/online_retail_II_2010-2011.csv"
df_2 = spark.read.csv(file_location_2, header=True, inferSchema=True)
df_2.show(10)

## Data Types

In [0]:
# Check data types
df.printSchema()

In [0]:
# Check data types
df_2.printSchema()

## Data Dimensions

In [0]:
# Check shape
print(df.count(), len(df.columns))
print(df_2.count(), len(df_2.columns))

In [0]:
# Combine 2 dataframes into one df
df = df.union(df_2)

In [0]:
from pyspark.sql.functions import to_timestamp

# Fix data type
df = df \
    .withColumn("InvoiceDate", to_timestamp("InvoiceDate", "dd/MM/yyyy HH:mm"))
print(df.count(), len(df.columns))
df.printSchema()

## Missing Values

In [0]:
from pyspark.sql.functions import col, count, isnan, when

# Check missing values
# Count NULL values in each column
df.select(
    [
        count(when(col(c).isNull(), c)).alias(c) 
        for c in df.columns
    ]
).show()

# Count NAN values in each column
df.select(
    [
        count(when(isnan(col(c)), c)).alias(c) 
        for c, dtype in df.dtypes
        if dtype == 'double'
    ]
).show()

## Duplicate

In [0]:
print(df.count() - df.dropDuplicates().count())

## Count Distinct Values

In [0]:
from pyspark.sql.functions import col, countDistinct

df.select(
    [
        countDistinct(col(c), ).alias(c) 
        for c in df.columns
    ]
).show()

## Check Invalid Values

In [0]:
from pyspark.sql.functions import col, count, when

df.select(
    count(when(col("Quantity") <= 0, 1)).alias("Invalid_Quantities"),
    count(when(col("Price") <= 0, 1)).alias("Invalid_Prices"),
    count(when(col("Customer ID") <= 0, 1)).alias("Negative_Customer_IDs")
).show()


In [0]:
from pyspark.sql.functions import col

# negative_rows_df = df.filter(
#     (col("Quantity") <= 0) | 
#     (col("Price") <= 0)
# )

negative_rows_df = df.filter(
    (col("Price") <= 0)
)
negative_rows_df.show()


In [0]:
# Check InvoiceDate range
from pyspark.sql.functions import min, max

df.select(
    min("InvoiceDate").alias("Earliest_Invoice_Date"),
    max("InvoiceDate").alias("Latest_Invoice_Date")
).show()

In [0]:
# Show all distinct values in the Country column
df.select("Country").distinct().show(43)